# Birdsong Classification with Machine Learning

In this notebook you will:

1. **Visualize** birdsong audio as waveforms and spectrograms
2. **Process** recordings from 5 bird species (Bachman's sparrow, Northern cardinal, Song sparrow, Swamp sparrow, Zebra finch)
3. **Extract features** using MFCCs (Mel Frequency Cepstral Coefficients)
4. **Cluster** the songs using an autoencoder that compresses each recording into a 2D point

By the end you will have a scatter plot showing how different species' songs group together based on their acoustic features.

## Why Birdsong?

- **Species monitoring** — automated audio analysis helps track bird populations and migration
- **Behavioral insights** — song patterns reveal information about mating, territory, and communication
- **Conservation** — detecting changes in vocalizations can signal environmental stress

No prior coding experience is needed. Just run each cell from top to bottom!

## How to Use This Notebook

- Click the **Play** button on the left side of each cell to run it, or press **Shift + Enter**
- Run cells **in order** from top to bottom — later cells depend on earlier ones
- If something goes wrong, try **Runtime → Restart and run all** from the menu

### Save Your Own Copy

1. Go to **File → Save a copy in Drive**
2. Rename it with your name (e.g. `YourName_Birdsong_Notebook.ipynb`)

## Step 1: Setup

This cell installs dependencies, imports libraries, and downloads the birdsong data from GitHub.

In [ ]:
# Install dependencies
!pip install -q pydub

# Import libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder, StandardScaler
from IPython.display import Audio, display

# Clone the birdsong dataset from GitHub
REPO_DIR = "/content/birdsong"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/williamedwardhahn/birdsong.git {REPO_DIR}
    print("Dataset downloaded!")
else:
    print("Dataset already downloaded.")

# Find all species folders (skip hidden dirs like .git)
SPECIES_FOLDERS = sorted([
    name for name in os.listdir(REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, name)) and not name.startswith(".")
])
print(f"\nFound {len(SPECIES_FOLDERS)} species: {SPECIES_FOLDERS}")

## Step 2: Helper Functions

These utility functions are used throughout the notebook:
- `plot_waveform` — shows amplitude over time
- `plot_spectrogram` — shows frequency content over time
- `Autoencoder` — a neural network that compresses features into 2 dimensions

In [ ]:
def plot_waveform(y, sr, title="Waveform"):
    """Plot the waveform of an audio signal."""
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y, sr=sr)
    plt.title(title)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()


def plot_spectrogram(y, sr, title="Spectrogram"):
    """Plot the spectrogram of an audio signal."""
    S = np.abs(librosa.stft(y))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.show()


class Autoencoder(nn.Module):
    """Compresses input features down to 2 dimensions, then reconstructs them."""
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded


print("Helper functions ready.")

## Section 1: Explore Individual Birdsong Files

Let's pick one sample from each species and look at its waveform, spectrogram, and listen to it.

In [ ]:
for species in SPECIES_FOLDERS:
    folder = os.path.join(REPO_DIR, species)
    wav_files = [f for f in os.listdir(folder) if f.endswith('.wav')]
    if not wav_files:
        continue

    # Pick the first file as a sample
    sample_file = os.path.join(folder, wav_files[0])
    y, sr = librosa.load(sample_file, sr=None)

    print(f"\n{'='*60}")
    print(f"Species: {species}")
    print(f"File: {wav_files[0]}  |  Sample rate: {sr} Hz  |  Duration: {len(y)/sr:.2f}s")
    print(f"{'='*60}")

    plot_waveform(y, sr, title=f"{species} — Waveform")
    plot_spectrogram(y, sr, title=f"{species} — Spectrogram")
    display(Audio(y, rate=sr))

## Section 2: Load All Species & Visualize Spectrograms

Now we load **every** WAV file from all 5 species, organize them by label, and display a montage of spectrograms for each species.

In [ ]:
# Load all audio files into tensors with species labels
data_list = []
label_list = []

label_encoder = LabelEncoder()
label_encoder.fit(SPECIES_FOLDERS)
label_decoder = dict(zip(label_encoder.transform(SPECIES_FOLDERS), SPECIES_FOLDERS))

for species in SPECIES_FOLDERS:
    folder = os.path.join(REPO_DIR, species)
    label = label_encoder.transform([species])[0]
    wav_files = sorted([f for f in os.listdir(folder) if f.endswith('.wav')])

    for fname in wav_files:
        fpath = os.path.join(folder, fname)
        try:
            audio, _ = librosa.load(fpath, sr=None, duration=10)
            data_list.append(torch.tensor(audio))
            label_list.append(label)
            print(f"  Loaded: {species}/{fname}")
        except Exception as e:
            print(f"  Error loading {fpath}: {e}")

# Pad all audio to the same length and stack into a single tensor
data_tensor = torch.nn.utils.rnn.pad_sequence(data_list, batch_first=True)
label_tensor = torch.tensor(label_list)
print(f"\nLoaded {len(data_list)} files. Tensor shape: {data_tensor.shape}")

In [ ]:
# Display a spectrogram montage for each species (up to 8 samples each)
SAMPLE_RATE = 22050  # librosa default used for mel spectrograms
MAX_PER_SPECIES = 8

for encoded_label in torch.unique(label_tensor).tolist():
    species_name = label_decoder[encoded_label]
    indices = (label_tensor == encoded_label).nonzero(as_tuple=True)[0]
    n = min(len(indices), MAX_PER_SPECIES)

    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    fig.suptitle(f"Spectrograms — {species_name}", fontsize=14)
    if n == 1:
        axes = [axes]

    for j in range(n):
        audio = data_tensor[indices[j].item()].numpy()
        S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=128, fmax=8000)
        S_db = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_db, sr=SAMPLE_RATE, ax=axes[j], x_axis='time', y_axis='mel')
        axes[j].set_title(f"#{j+1}", fontsize=9)
        axes[j].set_xticks([])
        axes[j].set_yticks([])

    plt.tight_layout()
    plt.show()

## Section 3: Feature Extraction & Clustering

Now we:
1. Extract **MFCC features** from each recording (13 coefficients → mean + std = 26 features per file)
2. Train an **autoencoder** to compress those 26 features into **2 dimensions**
3. Plot the 2D embeddings — each dot is one recording, colored by species

If the autoencoder works well, songs from the same species should cluster together.

In [ ]:
# Extract MFCC features from every recording
N_MFCC = 13
features = []
labels = []

for i in range(len(data_tensor)):
    audio = data_tensor[i].numpy()
    mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    feature_vec = np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])
    features.append(feature_vec)
    labels.append(label_tensor[i].item())

X = np.array(features)
y = np.array(labels)

# Standardize features to zero mean and unit variance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

print(f"Feature matrix shape: {X.shape}  (samples x features)")

In [ ]:
# Train the autoencoder
model = Autoencoder(input_dim=X_scaled.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

NUM_EPOCHS = 5000
model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    encoded, decoded = model(X_tensor)
    loss = criterion(decoded, X_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}  —  Loss: {loss.item():.4f}")

print("Training complete!")

In [ ]:
# Generate 2D embeddings and plot
model.eval()
with torch.no_grad():
    embeddings, _ = model(X_tensor)
embeddings = embeddings.numpy()

plt.figure(figsize=(10, 7))
for label in np.unique(y):
    mask = y == label
    plt.scatter(embeddings[mask, 0], embeddings[mask, 1],
                label=label_decoder[label], alpha=0.7, s=60)

plt.legend(fontsize=11)
plt.title("2D Embedding of Birdsong MFCC Features (Autoencoder)", fontsize=14)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Each dot is one recording. Songs from the same species should cluster together!")